# Week 2: Text Cleaning and Normalization

This notebook profiles raw MLS remarks, applies the `TextCleaner` pipeline, and documents before/after examples.

In [3]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "scripts").exists(): ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

from scripts.text_cleaning import TextCleaner

raw_df = pd.read_csv(ROOT / "data/processed/listing_sample.csv")
cleaned_df = pd.read_csv( ROOT / "data/processed/listing_sample_cleaned.csv")
cleaner = TextCleaner()
print(raw_df.shape)

(1000, 7)


## Raw-text profiling

The profile is computed before cleaning so each transformation corresponds to a real pattern in the data.

In [4]:
profile = cleaner.profile_column(raw_df, "remarks")
print(f"Null rate: {profile['null_rate']:.2%}")
print(f"Average length: {profile['avg_length']:.0f} characters")
print(f"HTML-like tags: {profile['has_html']}")
print(f"Compact price mentions: {profile['price_mentions']}")
print("Detected abbreviations:", ", ".join(profile['common_abbreviations']))
pd.DataFrame(profile['common_terms'], columns=['term', 'count'])

Null rate: 0.00%
Average length: 1337 characters
HTML-like tags: 1
Compact price mentions: 13
Detected abbreviations: br, bdrm, bd, ba, sqft, sq ft, sf, w/, w/o, a/c, ac, hvac, gar, kit, lr, dr, mbr, mb, hoa, ss, app, upd, renov, yr, yrs, approx., approx, incl, det


,term,count
0,and,9472
1,the,8605
2,with,3657
3,to,3417
4,of,2947
5,in,2926
6,home,2539
7,for,2447
8,this,2264
9,living,1920


## Before and after examples

Only rows where cleaning changed the text are displayed. The original `remarks` column is retained in the cleaned CSV for traceability.

In [5]:
examples = cleaned_df.loc[cleaned_df['remarks'] != cleaned_df['remarks_cleaned'], ['L_ListingID', 'remarks', 'remarks_cleaned']].head(10)
for _, row in examples.iterrows():
    print(f"Listing {row['L_ListingID']}")
    print("BEFORE:", row['remarks'][:500])
    print("AFTER: ", row['remarks_cleaned'][:500])
    print("-" * 80)

Listing 1157753806
BEFORE: Tenants in the front unit just moved out, and it is available for viewing!! Please DO NOT disturb tenants in the rear house. Two detached single-family homes on one lot. Ideal for an owner live in one and rents out the other house. This will essentially reduce the cost of home ownership by about half and can be financed with an FHA loan. It is located very close to San Bernardine Medical Center. The front home is big enough for a small or medium family. It offers 3 bedrooms and 1 bathroom (appro
AFTER:  Tenants in the front unit just moved out, and it is available for viewing!! Please DO NOT disturb tenants in the rear house. Two detached single-family homes on one lot. Ideal for an owner live in one and rents out the other house. This will essentially reduce the cost of home ownership by about half and can be financed with an FHA loan. It is located very close to San Bernardine Medical Center. The front home is big enough for a small or medium family. It off

## Cleaning pipeline

1. Unicode normalization
2. HTML removal and entity decoding
3. Price normalization (`450k` to `450000`)
4. Measurement normalization (`2,000 sqft` to `2000 square feet`)
5. Expansion of 30+ MLS abbreviations
6. Whitespace normalization

The cleaned text is intentionally not lowercased so proper names and sentence readability are retained for downstream entity extraction.

## Week 2 conclusion

The cleaned dataset is ready for Week 3 numeric entity extraction. The raw text remains unchanged, while `remarks_cleaned` provides a standardized version for regex and NLP processing.